# 04 Random forest

Same protocol as XGBoost: randomized search (40 candidates, 5-fold CV) inside each outer fold.
Not part of the 2024 report; included as a second tree-ensemble baseline.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline

from common import (FIGURES_DIR, INNER_CV, RESULTS_DIR, SEED, load_train, make_preprocessor,
                    nested_cv, save_results, summarize, write_submission)

X, y = load_train()
X.shape

(1504, 32)

In [2]:
from scipy.stats import randint
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

pipe = Pipeline([("prep", make_preprocessor()),
                 ("clf", RandomForestClassifier(random_state=SEED, n_jobs=1))])
space = {
    "clf__n_estimators": randint(100, 601),
    "clf__max_depth": [None, 4, 6, 8, 10, 12, 16, 20],
    "clf__min_samples_leaf": randint(1, 11),
    "clf__max_features": ["sqrt", "log2", 0.3, 0.5, 0.8],
}
search = RandomizedSearchCV(pipe, space, n_iter=40, scoring="accuracy", cv=INNER_CV,
                            random_state=SEED, n_jobs=-1)

## Nested cross-validation

In [3]:
folds, oof, params = nested_cv(search, X, y)
save_results("random_forest", folds, oof, params)
print(params.to_string(index=False))
summarize(folds)

 fold  clf__max_depth  clf__max_features  clf__min_samples_leaf  clf__n_estimators
    0            16.0                0.8                      4                542
    1             NaN                0.5                      3                407
    2            16.0                0.8                      4                542
    3            16.0                0.8                      4                542
    4             8.0                0.5                      4                182


accuracy         precision         recall         specificity  \
                 mean     std      mean     std   mean     std        mean   
rule                                                                         
0.5            0.9535  0.0135    0.9467  0.0298  0.921  0.0208      0.9712   
recall>=0.90   0.9535  0.0135    0.9467  0.0298  0.921  0.0208      0.9712   

                         fpr            fnr              f1             auc  \
                 std    mean     std   mean     std    mean     std    mean   
rule                                                                          
0.5           0.0173  0.0288  0.0173  0.079  0.0208  0.9334  0.0188  0.9561   
recall>=0.90  0.0173  0.0288  0.0173  0.079  0.0208  0.9334  0.0188  0.9561   

                      
                 std  
rule                  
0.5           0.0103  
recall>=0.90  0.0103

## Final model

In [4]:
final = search.fit(X, y)
print("chosen:", final.best_params_, f"| inner CV accuracy {final.best_score_:.4f}")
names = final.best_estimator_["prep"].get_feature_names_out()
pd.Series(final.best_estimator_["clf"].feature_importances_, index=names).sort_values(ascending=False).head(8).round(3)

chosen: {'clf__max_depth': 8, 'clf__max_features': 0.5, 'clf__min_samples_leaf': 3, 'clf__n_estimators': 283} | inner CV accuracy 0.9561


FunctionalAssessment        0.234
ADL                         0.217
MMSE                        0.198
MemoryComplaints            0.129
BehavioralProblems          0.095
CholesterolTriglycerides    0.014
DietQuality                 0.012
PhysicalActivity            0.011
dtype: float64

In [5]:
write_submission("random_forest", final)

PosixPath('submissions/random_forest.csv')